In [1]:
import os
import ray
import json
import pickle
from dynaconf import Dynaconf
from tqdm.notebook import tqdm
from utils import check_path, convert_np_arrays, flatten_dict, env_creator
from ray.tune.logger import JsonLogger
from ray.rllib.algorithms.a2c import A2C
from ray.tune.registry import register_env

pygame 2.5.2 (SDL 2.28.2, Python 3.9.16)
Hello from the pygame community. https://www.pygame.org/contribute.html


/home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/botocore/httpsession.py:41: DeprecationWarning: 'urllib3.contrib.pyopenssl' module is deprecated and will be removed in a future release of urllib3 2.x. Read more in this issue: https://github.com/urllib3/urllib3/issues/2680
  from urllib3.contrib.pyopenssl import orig_util_SSLContext as SSLContext
2024-04-24 18:25:28,698	WARNING __init__.py:10 -- A3C has/have been moved to `rllib_contrib` and will no longer be maintained by the RLlib team. You can still use it/them normally inside RLlib util Ray 2.8, but from Ray 2.9 on, all `rllib_contrib` algorithms will no longer be part of the core repo, and will therefore have to be installed separately with pinned dependencies for e.g. ray[rllib] and other packages! See https://github.com/ray-project/ray/tree/master/rllib_contrib#rllib-contrib for more information on the RLlib contrib effort.
2024-04-24 18:25:28,699	WARNING __init__.py:10 -- A2C has/have been moved to `rllib_

In [2]:
# Init Ray
ray.init(
    num_cpus=5, num_gpus=1,
    include_dashboard=False,
    _system_config={"maximum_gcs_destroyed_actor_cached_count": 20},
)

2024-04-24 18:25:31,837	INFO worker.py:1673 -- Started a local Ray instance.


Python version:,3.9.16
Ray version:,2.8.0


(pid=2252368) /home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/botocore/httpsession.py:41: DeprecationWarning: 'urllib3.contrib.pyopenssl' module is deprecated and will be removed in a future release of urllib3 2.x. Read more in this issue: https://github.com/urllib3/urllib3/issues/2680
(pid=2252368)   from urllib3.contrib.pyopenssl import orig_util_SSLContext as SSLContext
(RolloutWorker pid=2252368) 2024-04-24 18:25:35,157	WARNING __init__.py:10 -- A3C has/have been moved to `rllib_contrib` and will no longer be maintained by the RLlib team. You can still use it/them normally inside RLlib util Ray 2.8, but from Ray 2.9 on, all `rllib_contrib` algorithms will no longer be part of the core repo, and will therefore have to be installed separately with pinned dependencies for e.g. ray[rllib] and other packages! See https://github.com/ray-project/ray/tree/master/rllib_contrib#rllib-contrib for more information on the RLlib contrib effort.
(RolloutWorker pid=2252368) 

In [3]:
import mlflow
from mlflow.exceptions import MlflowException
from func_timeout import FunctionTimedOut
from botocore.exceptions import ConnectionClosedError

In [4]:
import datetime
# Config path
env_name = "MiniGrid-LavaCrossingS9N1"
run_name = int(datetime.datetime.now().timestamp())
log_path = "/home/seventheli/data/BER/experiments/logging/%s" % env_name
checkpoint_path = "/home/seventheli/data/BER/experiments/checkpoints/%s" % env_name
# Set mlflow
mlflow.set_tracking_uri("http://localhost:9999")
mlflow.set_experiment(experiment_name=env_name)
mlflow_client = mlflow.tracking.MlflowClient()

In [5]:
# Check path available
import shutil
check_path(log_path)
log_path = str(os.path.join(log_path, str(run_name)))
check_path(log_path)
if os.path.exists(log_path):
    shutil.rmtree(log_path)
check_path(log_path)
check_path(checkpoint_path)
checkpoint_path = os.path.join(checkpoint_path, str(run_name))
check_path(checkpoint_path)
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
check_path(checkpoint_path)

In [6]:
# Set hyper parameters
setting = "./settings/a2c_atari.yml"
setting = Dynaconf(envvar_prefix="DYNACONF", settings_files=setting)

hyper_parameters = setting.hyper_parameters.to_dict()
hyper_parameters["logger_config"] = {"type": JsonLogger, "logdir": checkpoint_path}
hyper_parameters["env_config"] = {"id": env_name, "tile_size": 10, "img_size": 84, "max_steps": 100}

In [7]:
# Set run object
run_name = "A2C_%s" % env_name + "_%s" % run_name
env_example = env_creator(hyper_parameters["env_config"])
obs, _ = env_example.reset()
step = env_example.step(1)
print(env_example.action_space, env_example.observation_space)
print(env_example)
print("log path: %s; check_path: %s" % (log_path, checkpoint_path))
register_env("MiniGrid", env_creator)

Discrete(7) Box(0, 255, (84, 84, 3), uint8)
<TimeLimit<ResizeObservation<ImgObsWrapper<RGBImgObsWrapper<OrderEnforcing<PassiveEnvCheckerWGWGWGWGWGWGWGWGWG
WGVV            WG
WG              WG
WG              WG
WG              WG
WG              WG
WGVRVRVRVRVR  VRWG
WG            GGWG
WGWGWGWGWGWGWGWGWG>>>>>>
log path: /home/seventheli/data/BER/experiments/logging/MiniGrid-LavaCrossingS9N1/1713979532; check_path: /home/seventheli/data/BER/experiments/checkpoints/MiniGrid-LavaCrossingS9N1/1713979532


/home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/gymnasium/envs/registration.py:531: UserWarning: WARN: Using the latest versioned environment `MiniGrid-LavaCrossingS9N1-v0` instead of the unversioned environment `MiniGrid-LavaCrossingS9N1`.
  logger.warn(


In [8]:
# Set Trainer
trainer = A2C(config=hyper_parameters, env="MiniGrid")

2024-04-24 18:25:32,896	WARNING deprecation.py:50 -- DeprecationWarning: `rllib/algorithms/a2c/` has been deprecated. Use `rllib_contrib/a2c/` instead. This will raise an error in the future!
2024-04-24 18:25:32,898	WARNING deprecation.py:50 -- DeprecationWarning: `rllib/algorithms/a3c/` has been deprecated. Use `rllib_contrib/a3c/` instead. This will raise an error in the future!
2024-04-24 18:25:32,900	WARNING deprecation.py:50 -- DeprecationWarning: `algo = Algorithm(env='MiniGrid', ...)` has been deprecated. Use `algo = AlgorithmConfig().environment('MiniGrid').build()` instead. This will raise an error in the future!
/home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/ray/rllib/utils/from_config.py:197: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.Jso

In [9]:
# Common setup
checkpoint_path = str(checkpoint_path)
# Save initial configuration
with open(os.path.join(checkpoint_path, "%s_config.pyl" % run_name), "wb") as f:
    _ = trainer.config.to_dict()
    pickle.dump(_, f)

In [10]:
mlflow_run = mlflow.start_run(run_name=run_name,
                              tags={"mlflow.user": "Local"})
# Log parameters
to_log = ['num_workers',]
mlflow.log_params(
    {key: hyper_parameters[key] for key in to_log})

In [11]:
keys_to_extract_sam = {"episode_reward_max", "episode_reward_min", "episode_reward_mean"}
keys_to_extract_sta = {"num_agent_steps_sampled", "num_agent_steps_trained"}

In [12]:
mlflow.pytorch.log_model(trainer.get_policy().model, run_name)
model_uri = "runs:/%s/model_name" % mlflow_run.info.run_id
mlflow.register_model(model_uri, run_name, tags={"episode" : 0})

/home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'A2C_MiniGrid-LavaCrossingS9N1_1713979532'.
2024/04/24 18:25:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: A2C_MiniGrid-LavaCrossingS9N1_1713979532, version 1
Created version '1' of model 'A2C_MiniGrid-LavaCrossingS9N1_1713979532'.


<ModelVersion: aliases=[], creation_timestamp=1713979544042, current_stage='None', description='', last_updated_timestamp=1713979544042, name='A2C_MiniGrid-LavaCrossingS9N1_1713979532', run_id='0bf62aac7070409a8fe42babfef750e9', run_link='', source='s3://jo-mlflow-ber/29/0bf62aac7070409a8fe42babfef750e9/artifacts/model_name', status='READY', status_message='', tags={'episode': '0'}, user_id='', version='1'>

In [ ]:
for i in tqdm(range(0, 100000), ascii=True):
    result = trainer.train()
    time_used = result["time_total_s"]
    try:
        sampler = result.get("sampler_results", {}).copy()
        eva = result.get("evaluation", {}).copy()
        info = result.get("info", {}).copy()
        sam = {key: sampler[key] for key in keys_to_extract_sam if key in sampler}
        sta = {key: info[key] for key in keys_to_extract_sta if key in info}
        if eva:
            eva = {"eval_" + key: eva[key] for key in keys_to_extract_sam if key in eva}
        mlflow.log_metrics({**sam, **sta, **eva}, step=result["episodes_total"])
        if i % 50 == 0:
            trainer.save_checkpoint(checkpoint_path)
            mlflow.pytorch.log_model(trainer.get_policy().model, run_name)
            mlflow.register_model(model_uri, run_name, tags={
                "episode" : result["episodes_total"],
                "reward": sam["episode_reward_mean"],
            })
            mlflow.log_artifacts(log_path)
            mlflow.log_artifacts(checkpoint_path)
        if i % 10 == 0:
            tqdm.write("episode %d ; " % result["episodes_total"] +  " ".join(["%s : %f8" % (i, j)for i, j in sam.items()]))
    except FunctionTimedOut:
        tqdm.write("logging failed")
    except MlflowException:
        tqdm.write("logging failed")
    except ConnectionClosedError:
        tqdm.write("logging failed")
    with open(os.path.join(log_path, str(i) + ".json"), "w") as f:
        result["config"] = None
        json.dump(convert_np_arrays(result), f)

  0%|          | 0/100000 [00:00<?, ?it/s]

2024-04-24 18:25:44,236	WARNING deprecation.py:50 -- DeprecationWarning: `ray.rllib.execution.train_ops.multi_gpu_train_one_step` has been deprecated. This will raise an error in the future!
Registered model 'A2C_MiniGrid-LavaCrossingS9N1_1713979532' already exists. Creating a new version of this model...
2024/04/24 18:25:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: A2C_MiniGrid-LavaCrossingS9N1_1713979532, version 2
Created version '2' of model 'A2C_MiniGrid-LavaCrossingS9N1_1713979532'.


episode 30 ; episode_reward_mean : 0.0000008 episode_reward_max : 0.0000008 episode_reward_min : 0.0000008


In [ ]:
mlflow.log_artifacts(log_path)
mlflow.log_artifacts(checkpoint_path)

In [ ]:
mlflow.end_run()